# LLM Baseline — Real-Time Object Detection on KITTI

Evaluates LLM-based object detection as a baseline against YOLO26n + BNN.

| Section | Model |
|---------|-------|
| A | LLaVA-1.5-7B (local GPU) |
| B | Gemini 2.0 Flash (API) |
| C | Gemini 2.5 Flash (API) |
| D | ChatGPT GPT-4o (API) |
| E | Demo — 5 images inline |
| F | Results comparison table |

---
## 0. Setup
Run once at the start of every Colab session.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠ No GPU — Runtime → Change runtime type → T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

In [ ]:
!pip install -q transformers accelerate bitsandbytes pillow
!pip install -q google-genai
!pip install -q openai

In [ ]:
import os, sys
REPO   = 'Introduction-to-Artificial-Intelligence-Final-Project'
BRANCH = 'claude/musing-brahmagupta-7d497e'
if not os.path.exists(f'/content/{REPO}'):
    !git clone https://github.com/Appledog3572/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git checkout {BRANCH} -q
!git pull -q
sys.path.insert(0, 'llava_baseline')
print('Repo ready.')

---
## 1. API Keys
Fill in your keys. Leave empty to skip that model.

In [ ]:
# === Fill in your API keys ===
GEMINI_API_KEY = ''   # Google AI Studio → Get API key
OPENAI_API_KEY = ''   # platform.openai.com → API keys
HF_TOKEN       = ''   # huggingface.co/settings/tokens (speeds up LLaVA download)

# Switch: True = all 8 KITTI classes, False = Car / Pedestrian / Cyclist only
FULL_CLASSES = False

# ---
import os
if HF_TOKEN:       os.environ['HF_TOKEN']       = HF_TOKEN
if GEMINI_API_KEY: os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
if OPENAI_API_KEY: os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('Keys configured. FULL_CLASSES =', FULL_CLASSES)

---
## 2. Shared helpers (demo + display)

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from dataset import KITTIDataset, CLASS_NAMES
from infer import InferenceRunner

# Fixed demo image IDs (val split)
DEMO_IDS = ['000001', '000013', '000017', '000035', '000000']

GT_COLOR   = (0, 200, 0)
PRED_COLOR = (220, 30, 30)


def _draw_boxes(image, gt_list, pred_list):
    img  = image.copy()
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('arial.ttf', 13)
    except Exception:
        font = ImageFont.load_default()
    for objs, color, prefix in [(gt_list, GT_COLOR, 'GT'), (pred_list, PRED_COLOR, 'PR')]:
        for obj in objs:
            x1, y1, x2, y2 = obj['bbox']
            cid   = obj.get('class_id', -1)
            label = f"{prefix}:{cid}"
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            bb = draw.textbbox((x1, y1), label, font=font)
            tw, th = bb[2]-bb[0], bb[3]-bb[1]
            ty = y1-th-3 if y1-th-3 >= 0 else y1+2
            draw.rectangle([x1, ty, x1+tw+4, ty+th+4], fill=color)
            draw.text((x1+2, ty+2), label, fill=(255,255,255), font=font)
    return img


def run_demo(runner, title):
    """Run inference on DEMO_IDS and display inline."""
    dataset = KITTIDataset('datasets/kitti_dataset', split='val',
                           full_classes=FULL_CLASSES)
    id_map  = {item['image_path'].stem: item for item in dataset}

    fig, axes = plt.subplots(len(DEMO_IDS), 1, figsize=(16, 4*len(DEMO_IDS)))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for ax, img_id in zip(axes, DEMO_IDS):
        item = id_map.get(img_id)
        if item is None:
            ax.set_title(f'{img_id} not found'); ax.axis('off'); continue
        result = runner.run(item['image'])
        gt_list   = [{'class_id': g['class_id'], 'bbox': g['bbox']} for g in item['gt']]
        pred_list = [{'class_id': d.class_id,    'bbox': d.bbox}    for d in result.detections]
        vis = _draw_boxes(item['image'], gt_list, pred_list)
        ax.imshow(vis)
        ax.set_title(f"{img_id}  GT:{len(gt_list)}  PR:{len(pred_list)}  "
                     f"{result.latency_ms:.0f}ms", fontsize=11)
        ax.axis('off')

    g_patch = mpatches.Patch(color=(0,200/255,0), label='Ground Truth')
    r_patch = mpatches.Patch(color=(220/255,30/255,30/255), label='Prediction')
    fig.legend(handles=[g_patch, r_patch], loc='lower right', fontsize=11)
    plt.tight_layout(); plt.show()


def print_summary(results):
    print(f"  mAP@0.5:       {results['mAP']:.4f}")
    for cls, ap in results['AP_per_class'].items():
        print(f"    {cls:<16s} AP={ap:.4f}")
    print(f"  Mean latency:  {results['mean_latency_ms']:.1f} ms")
    print(f"  Mean FPS:      {results['mean_FPS']:.2f}")
    print(f"  Model size:    {results['model_size_MB']:.1f} MB")


print('Helpers loaded.')

---
# A. LLaVA-1.5-7B
Local model — requires T4 GPU.

In [ ]:
FC = '--full-classes' if FULL_CLASSES else ''
!mkdir -p results/llava/vis
!python llava_baseline/evaluate.py \
    --split val --mode llava --max-images 100 \
    {FC} \
    --vis-dir results/llava/vis \
    --output-json results/llava/val.json

In [ ]:
llava_r = json.loads(Path('results/llava/val.json').read_text())
print('=== LLaVA-1.5-7B ==='); print_summary(llava_r)

In [ ]:
# Show confusion matrix inline
from IPython.display import Image as IPyImage
IPyImage('results/llava/val_confusion.png')

In [ ]:
# Demo — 5 fixed images inline
llava_runner = InferenceRunner(mode='llava')
run_demo(llava_runner, 'LLaVA-1.5-7B  |  Green=GT  Red=Pred  Label=ClassID')

In [ ]:
# Save to Drive
import shutil
shutil.copytree('results/llava', '/content/drive/MyDrive/llm_results/llava',
                dirs_exist_ok=True)
print('Saved to Drive: llm_results/llava')

---
# B. Gemini 2.0 Flash
Fill in `GEMINI_API_KEY` in Section 1.

In [ ]:
FC = '--full-classes' if FULL_CLASSES else ''
!mkdir -p results/gemini20/vis
!python llava_baseline/evaluate.py \
    --split val --mode gemini --model-id gemini-2.0-flash --max-images 100 \
    {FC} \
    --vis-dir results/gemini20/vis \
    --api-key "$GEMINI_API_KEY" \
    --output-json results/gemini20/val.json

In [ ]:
g20_r = json.loads(Path('results/gemini20/val.json').read_text())
print('=== Gemini 2.0 Flash ==='); print_summary(g20_r)

In [ ]:
from IPython.display import Image as IPyImage
IPyImage('results/gemini20/val_confusion.png')

In [ ]:
g20_runner = InferenceRunner(mode='gemini', model_id='gemini-2.0-flash',
                              api_key=GEMINI_API_KEY)
run_demo(g20_runner, 'Gemini 2.0 Flash  |  Green=GT  Red=Pred  Label=ClassID')

In [ ]:
import shutil
shutil.copytree('results/gemini20', '/content/drive/MyDrive/llm_results/gemini20',
                dirs_exist_ok=True)
print('Saved to Drive: llm_results/gemini20')

---
# C. Gemini 2.5 Flash
Same key as Gemini 2.0.

In [ ]:
FC = '--full-classes' if FULL_CLASSES else ''
!mkdir -p results/gemini25/vis
!python llava_baseline/evaluate.py \
    --split val --mode gemini --model-id gemini-2.5-flash --max-images 100 \
    {FC} \
    --vis-dir results/gemini25/vis \
    --api-key "$GEMINI_API_KEY" \
    --output-json results/gemini25/val.json

In [ ]:
g25_r = json.loads(Path('results/gemini25/val.json').read_text())
print('=== Gemini 2.5 Flash ==='); print_summary(g25_r)

In [ ]:
from IPython.display import Image as IPyImage
IPyImage('results/gemini25/val_confusion.png')

In [ ]:
g25_runner = InferenceRunner(mode='gemini', model_id='gemini-2.5-flash',
                              api_key=GEMINI_API_KEY)
run_demo(g25_runner, 'Gemini 2.5 Flash  |  Green=GT  Red=Pred  Label=ClassID')

In [ ]:
import shutil
shutil.copytree('results/gemini25', '/content/drive/MyDrive/llm_results/gemini25',
                dirs_exist_ok=True)
print('Saved to Drive: llm_results/gemini25')

---
# D. ChatGPT GPT-4o
Fill in `OPENAI_API_KEY` in Section 1.

In [ ]:
FC = '--full-classes' if FULL_CLASSES else ''
!mkdir -p results/gpt4o/vis
!python llava_baseline/evaluate.py \
    --split val --mode gpt4o --model-id gpt-4o --max-images 100 \
    {FC} \
    --vis-dir results/gpt4o/vis \
    --api-key "$OPENAI_API_KEY" \
    --output-json results/gpt4o/val.json

In [ ]:
gpt_r = json.loads(Path('results/gpt4o/val.json').read_text())
print('=== GPT-4o ==='); print_summary(gpt_r)

In [ ]:
from IPython.display import Image as IPyImage
IPyImage('results/gpt4o/val_confusion.png')

In [ ]:
gpt_runner = InferenceRunner(mode='gpt4o', model_id='gpt-4o',
                              api_key=OPENAI_API_KEY)
run_demo(gpt_runner, 'GPT-4o  |  Green=GT  Red=Pred  Label=ClassID')

In [ ]:
import shutil
shutil.copytree('results/gpt4o', '/content/drive/MyDrive/llm_results/gpt4o',
                dirs_exist_ok=True)
print('Saved to Drive: llm_results/gpt4o')

---
# F. Results Comparison

In [ ]:
import json
from pathlib import Path

result_files = {
    'LLaVA-1.5-7B':     'results/llava/val.json',
    'Gemini 2.0 Flash': 'results/gemini20/val.json',
    'Gemini 2.5 Flash': 'results/gemini25/val.json',
    'GPT-4o':           'results/gpt4o/val.json',
}

rows = []
for name, path in result_files.items():
    if not Path(path).exists(): continue
    r = json.loads(Path(path).read_text())
    rows.append({'Model': name,
                 'mAP@0.5':    r['mAP'],
                 'FPS':        r['mean_FPS'],
                 'Lat(ms)':    r['mean_latency_ms'],
                 'Size(MB)':   r['model_size_MB'],
                 'N':          r['n_images']})

if rows:
    keys  = list(rows[0].keys())
    col_w = [max(len(k), max(len(str(r[k])) for r in rows))+2 for k in keys]
    sep   = '+' + '+'.join('-'*w for w in col_w) + '+'
    fmt   = '|' + '|'.join(f'{{:<{w}}}' for w in col_w) + '|'
    print(sep)
    print(fmt.format(*keys))
    print(sep)
    for r in rows:
        print(fmt.format(*[str(r[k]) for k in keys]))
    print(sep)
else:
    print('No results found — run evaluation cells first.')